# Tiny GPT2 like Transformer

In this notebook, we train a small GPT2-style Transformer on the TinyStories dataset, then use it to write a story.

The slow steps save their results to files (`tokenizer.json`, `train_ids.npy`, `val_ids.npy`), so later runs skip them. 

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from tqdm.auto import tqdm

device = "cuda"
torch.set_float32_matmul_precision("high")  # faster math on RTX GPUs

# Settings
batch_size = 64     # stories snippets per training step
block_size = 256    # how many tokens the model sees at once
n_embd = 384        # size of each token's vector
n_head = 6          # attention heads per layer
n_layer = 6         # number of Transformer layers
max_steps = 2000
learning_rate = 3e-4

c:\Users\Dhananjay Lakra\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the stories

We keep the last 5% of stories aside as validation data, to check the model on stories it never trained on.

In [2]:
texts = list(load_dataset("karpathy/tinystories-gpt4-clean")["train"]["text"])

split = int(0.95 * len(texts))
train_texts = texts[:split]
val_texts = texts[split:]

print(f"Training stories:   {len(train_texts):,}")
print(f"Validation stories: {len(val_texts):,}")

Training stories:   2,596,002
Validation stories: 136,632


## 2. Train a tokenizer

A tokenizer turns text into numbers. BPE starts with single bytes and keeps merging the most common neighboring pairs until it has 8,000 tokens. `[EOS]` marks the end of a story.

In [3]:
if os.path.exists("tokenizer.json"):
    enc = Tokenizer.from_file("tokenizer.json")
else:
    enc = Tokenizer(models.BPE(unk_token="[UNK]"))
    enc.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    enc.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=8000,
        min_frequency=2,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    )
    enc.train_from_iterator(train_texts, trainer=trainer)
    enc.save("tokenizer.json")

vocab_size = enc.get_vocab_size()
eos_id = enc.token_to_id("[EOS]")

print("Vocabulary size:", vocab_size)
print(enc.encode("Once upon a time").tokens)

Vocabulary size: 8000
['Once', 'Ġupon', 'Ġa', 'Ġtime']


## 3. Turn every story into tokens

All stories are joined into one long list of token ids, with `[EOS]` after each story.

In [ ]:
def encode_stories(stories, file_name):
    if os.path.exists(file_name):
        return np.load(file_name)

    pieces = []
    for i in tqdm(range(0, len(stories), 20_000), desc=file_name):
        for e in enc.encode_batch(stories[i:i + 20_000]):
            pieces.append(np.array(e.ids + [eos_id], dtype=np.uint16))

    ids = np.concatenate(pieces)
    np.save(file_name, ids)
    return ids

train_ids = encode_stories(train_texts, "train_ids.npy")
val_ids = encode_stories(val_texts, "val_ids.npy")

# Move all tokens to the GPU 
train_data = torch.from_numpy(train_ids.astype(np.int32)).to(device)
val_data = torch.from_numpy(val_ids.astype(np.int32)).to(device)

print(f"Training tokens:   {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

Training tokens:   510,853,192
Validation tokens: 26,899,925


## 4. Make training batches

We cut random snippets out of the token list. The target `y` is the input `x` shifted one token to the right, because the model's job is to predict the next token.

In [5]:
def get_batch(data):
    starts = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in starts])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in starts])
    return x.long(), y.long()

x, y = get_batch(train_data)
print(x.shape, y.shape)

torch.Size([64, 256]) torch.Size([64, 256])


## 5. The model

**Head:** each token makes a query ("what am I looking for?"), a key ("what do I contain?") and a value ("what do I share?"). Query-key scores say how much each token should look at each earlier token. The mask hides future tokens so the model can't peek at the answer.

**Attention:** several heads run side by side and their results are joined.

**Block:** attention, then a small feed-forward network (MLP). `x + ...` (residual connections) and `LayerNorm` keep training stable.

**TinyTransformer:** token and position embeddings, a stack of blocks, then a final layer that scores every possible next token.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))) #Causal mask

    def forward(self, x):
        length = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5)
        scores = scores.masked_fill(self.mask[:length, :length] == 0, float("-inf"))
        return F.softmax(scores, dim=-1) @ v

class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd // n_head) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        joined = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.projection(joined)

class Block(nn.Module):  
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attention = Attention()
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))      
        x = x + self.mlp(self.norm2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(vocab_size, n_embd)        # token id -> vector
        self.positions = nn.Embedding(block_size, n_embd)     # position -> vector
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.norm = nn.LayerNorm(n_embd)
        self.output = nn.Linear(n_embd, vocab_size)           # vector -> score for every token

    def forward(self, ids):
        positions = torch.arange(ids.shape[1], device=ids.device)
        x = self.tokens(ids) + self.positions(positions)
        x = self.blocks(x)
        return self.output(self.norm(x))

model = TinyTransformer().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 16,890,944


## 6. Train
 AdamW is used instead of plain SGD because Transformers train much better with it. If validation loss starts rising while training loss keeps falling, the model is overfitting.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
def get_loss(x, y):
    logits = model(x)
    return F.cross_entropy(logits.view(-1, vocab_size), y.view(-1)) #cross entropy loss

for step in range(max_steps):
    x, y = get_batch(train_data)
    loss = get_loss(x, y)

    optimizer.zero_grad() #setting gradient = 0
    loss.backward()       #backpropagation
    optimizer.step()      #taking next step and updating the weights

    if step % 100 == 0:
        with torch.no_grad():
            val_loss = get_loss(*get_batch(val_data))
        print(f"step {step:5d} | train {loss.item():.3f} | validation {val_loss.item():.3f}")

step     0 | train 1.716 | validation 1.662
step   100 | train 1.808 | validation 1.772
step   200 | train 1.752 | validation 1.762
step   300 | train 1.656 | validation 1.720
step   400 | train 1.660 | validation 1.751
step   500 | train 1.734 | validation 1.729
step   600 | train 1.655 | validation 1.778
step   700 | train 1.754 | validation 1.721
step   800 | train 1.693 | validation 1.762
step   900 | train 1.705 | validation 1.660
step  1000 | train 1.623 | validation 1.697
step  1100 | train 1.669 | validation 1.734
step  1200 | train 1.716 | validation 1.771
step  1300 | train 1.693 | validation 1.723
step  1400 | train 1.615 | validation 1.633
step  1500 | train 1.701 | validation 1.632
step  1600 | train 1.717 | validation 1.672
step  1700 | train 1.616 | validation 1.671
step  1800 | train 1.696 | validation 1.652
step  1900 | train 1.676 | validation 1.703


## 7. Writing a story

In [13]:
ids = torch.tensor([enc.encode("Once upon a time").ids], device=device)

with torch.no_grad():
    for _ in range(200):
        logits = model(ids[:, -block_size:])
        probs = F.softmax(logits[:, -1], dim=-1)
        next_id = torch.multinomial(probs, 1)
        ids = torch.cat([ids, next_id], dim=1)

print(enc.decode(ids[0].tolist()))

Once upon a time, there was a little kitten named Zip. Zip loved to play and roll in the water. One day, Zip went outside to explore his home.
When Zip got dirty, he met a new friend, a big deaf dog named Grumpy. Grumpy was sad because he could not find any junk to play with. Zip wanted to help his new friend, so he decided to follow Whistle to the rightful place.
As Wink went crossing, he met a big cat. The cat was wearing a big apron with a lot of things on it. It was very unexpected! Zip was so happy to see Kit and didn't know which one to play with. Max and Beth played in the little house all day, and they became best friends.Once upon a time, in a big forest, there lived a creative lady named Amy. She loved to do things with her leaves. One day, She wanted to see what she could make with her leaves. So, she asked her friend, Max
